# Differential Equations — Session 30
## Section 7.1: Definition of the Laplace Transform

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to define the Laplace transform as an improper integral; determine its domain of convergence; use linearity; compute transforms of basic functions; state sufficient existence conditions; and interpret how the parameter $s$ weights early and late values of a function.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Integral transforms and improper integrals |
| 18–38 min | Direct calculations from the definition |
| 38–55 min | Linearity and transform table |
| 55–73 min | Piecewise continuity and exponential order |
| 73–86 min | Numerical transform experiments |
| 86–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import quad, solve_ivp
from scipy.signal import fftconvolve
from scipy.linalg import expm
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def unit_step(t, a=0.0):
    t = np.asarray(t)
    return (t >= a).astype(float)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 7.1-A — Laplace transform

For a function $f$ defined on $t\ge0$,

$$
\mathcal{L}\{f(t)\}
=
F(s)
=
\int_0^\infty e^{-st}f(t)\,dt,
$$

whenever the improper integral converges.

### Theorem 7.1-B — Linearity

If the transforms exist, then

$$
\mathcal{L}\{\alpha f+\beta g\}
=
\alpha\mathcal{L}\{f\}
+
\beta\mathcal{L}\{g\}.
$$

### Definition 7.1-C — Piecewise continuity

A function is piecewise continuous on every finite interval when it is continuous except at finitely many points and has finite one-sided limits at each discontinuity.

### Definition 7.1-D — Exponential order

A function is of exponential order $c$ if there exist $M,T>0$ such that

$$
|f(t)|\le Me^{ct}
$$

for all $t>T$.

### Theorem 7.1-E — Sufficient existence conditions

If $f$ is piecewise continuous on every finite interval and is of exponential order $c$, then $\mathcal{L}\{f\}(s)$ exists for $s>c$.

### Theorem 7.1-F — Basic transforms

For suitable $s$,

$$
\mathcal{L}\{1\}=\frac1s,
\qquad
\mathcal{L}\{t^n\}=\frac{n!}{s^{n+1}},
$$

$$
\mathcal{L}\{e^{at}\}=\frac1{s-a},
$$

$$
\mathcal{L}\{\cos bt\}=\frac{s}{s^2+b^2},
\qquad
\mathcal{L}\{\sin bt\}=\frac{b}{s^2+b^2}.
$$

### Classroom Checkpoint — Transform Convergence

If $f(t)$ is of exponential order $c$, for which real $s$ is its Laplace transform guaranteed to exist?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. The kernel $e^{-st}$

For $s>0$, the factor $e^{-st}$ suppresses late-time contributions. Larger $s$ concentrates the transform more strongly near $t=0$.

In [ ]:
t = np.linspace(0, 8, 700)
for s in [0.25, 0.5, 1, 2]:
    plt.plot(t, np.exp(-s*t), label=fr"$s={s}$")
plt.xlabel("t")
plt.ylabel(r"$e^{-st}$")
plt.title("The Laplace kernel")
plt.legend()
plt.show()

## 2. Direct calculation from the definition

For $f(t)=1$,

$$
F(s)=\int_0^\infty e^{-st}\,dt=\frac1s,
\qquad s>0.
$$

For $f(t)=e^{at}$,

$$
F(s)=\int_0^\infty e^{-(s-a)t}\,dt
=\frac1{s-a},
\qquad s>a.
$$

In [ ]:
def truncated_transform_constant(s=1.0, upper=8.0):
    value = quad(lambda t: np.exp(-s*t), 0, upper)[0]
    exact = 1/s if s > 0 else np.nan
    print("truncated integral:", value)
    print("exact transform:", exact)
    print("tail error:", abs(exact-value) if s > 0 else "divergent")

if WIDGETS_AVAILABLE:
    interact(
        truncated_transform_constant,
        s=FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0),
        upper=FloatSlider(min=1, max=30, step=1, value=8)
    )
else:
    truncated_transform_constant()

## 3. Domain of convergence

For $f(t)=e^{3t}$, the transform converges only when $s>3$.

In [ ]:
def growth_transform_scan(a=3.0, upper=10.0):
    s_values = np.linspace(a-2, a+4, 500)
    values = []
    for s in s_values:
        exponent = a-s
        if abs(exponent) < 1e-12:
            values.append(upper)
        else:
            values.append((np.exp(exponent*upper)-1)/exponent)
    plt.semilogy(s_values, np.abs(values))
    plt.axvline(a, linestyle="--", label=fr"boundary $s={a}$")
    plt.xlabel("s")
    plt.ylabel("absolute truncated integral")
    plt.title(fr"$\int_0^{{{upper}}}e^{{-(s-a)t}}dt$")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        growth_transform_scan,
        a=FloatSlider(min=-2, max=5, step=0.25, value=3),
        upper=FloatSlider(min=2, max=20, step=1, value=10)
    )
else:
    growth_transform_scan()

## 4. Piecewise function

Consider

$$
f(t)=
\begin{cases}
t,&0\le t<2,\\
2,&t\ge2.
\end{cases}
$$

Then

$$
\mathcal{L}\{f\}
=
\int_0^2 te^{-st}\,dt
+
\int_2^\infty 2e^{-st}\,dt.
$$

In [ ]:
s = sp.symbols("s", positive=True)
t = sp.symbols("t", nonnegative=True)
F_piece = sp.integrate(t*sp.exp(-s*t), (t, 0, 2)) + \
          sp.integrate(2*sp.exp(-s*t), (t, 2, sp.oo))
display(sp.simplify(F_piece))

In [ ]:
t_grid = np.linspace(0, 8, 600)
f = np.where(t_grid < 2, t_grid, 2)
plt.plot(t_grid, f)
plt.xlabel("t")
plt.ylabel("f(t)")
plt.title("A piecewise-continuous function")
plt.show()

## 5. Numerical transform as a function of $s$

Take $f(t)=t e^{-t}\sin(3t)$. Numerical quadrature lets students see the transform as a smooth function of $s$.

In [ ]:
def numerical_laplace(f, s_values, upper=40):
    return np.array([
        quad(lambda t: np.exp(-s*t)*f(t), 0, upper, limit=300)[0]
        for s in s_values
    ])

s_values = np.linspace(0.05, 6, 250)
F_values = numerical_laplace(lambda t: t*np.exp(-t)*np.sin(3*t), s_values)

plt.plot(s_values, F_values)
plt.xlabel("s")
plt.ylabel("F(s)")
plt.title(r"Numerical transform of $te^{-t}\sin(3t)$")
plt.show()

## 6. Large-$s$ behavior

Under standard existence assumptions, $F(s)\to0$ as $s\to\infty$. This reflects increasingly strong damping by the kernel.

In [ ]:
s_values = np.linspace(0.2, 20, 500)
transforms = {
    "1": 1/s_values,
    "t": 1/s_values**2,
    "sin(2t)": 2/(s_values**2+4),
}
for label, values in transforms.items():
    plt.plot(s_values, values, label=label)
plt.xlabel("s")
plt.ylabel("F(s)")
plt.title("Basic transforms decay for large s")
plt.legend()
plt.show()

## Classroom Checkpoint — Exit Check

Find the transform and convergence condition for $f(t)=e^{-4t}$.

> Pause here. Let students commit to an answer before running the next cell.